In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr




In [10]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
      print(f"openai string found")
else:
      print(f"open ai key not ffound")

MODEL = "gpt-4.1-nano"
DB_NAME= "vector_db"


openai string found


In [11]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings) #where are the documents 

In [12]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [13]:
relevantChunks = retriever.invoke("Who is Avery?")
print(relevantChunks)

[Document(id='b5ad20e4-8772-4f5b-a2b1-566f250ce0ac', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [15]:
simpleOpenAI = llm.invoke("who is avery")
print(simpleOpenAI)


content='Avery is a given name that can be used for both males and females. It can also be a surname. The meaning of the name Avery is often associated with "ruler of the elves" or "wise." If you\'re referring to a specific person named Avery, could you please provide more context or details?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 11, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'id': 'chatcmpl-CkkEdlxLpQKTBAARwSytVML79tPvg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--4581a12b-8d41-4075-9c3e-f70b9f13fab4-0' usage_metadata={'input_tokens': 11, 'output_tokens': 63, 'total_tokens': 74, 'input_

In [20]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
"""
USER_PROMPT_TEMPLATE = """
QUESTION: {question}
context: {retrieved}
"""

In [28]:
def answer_question (question: str, history):
      retrievedChucks = retriever.invoke(question)
      context = "\n\n".join(doc.page_content for doc in retrievedChucks)
      # add this to user info 
      user_message=USER_PROMPT_TEMPLATE.format(question=question, retrieved=context)
      # make the python call
      response = llm.invoke([SystemMessage(content=SYSTEM_PROMPT_TEMPLATE),HumanMessage(content=user_message)])
      return response.content

In [29]:
answer_question("who is avery", [])

'Avery Lancaster is the Co-Founder and CEO of Insurellm, based in San Francisco, California. She has been with the company since its founding in 2015 and has played a key role in its growth and success as a leading provider in the insurance technology industry. Avery is recognized for her innovative leadership, risk management expertise, and her active involvement in professional development, diversity initiatives, and community outreach.'

In [30]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
